# Generación de Ground Truth con DeepDoctection

Este notebook extrae tablas de los PDFs usando **deepdoctection** y genera un archivo JSON con el mismo formato del ground truth manual (`ground_truth_kge.json`).

El resultado está pensado para ser revisado y ajustado manualmente según las instrucciones en `ground_truth_manual.md`.

**Formato de salida:**
```json
{
  "documents": [
    {
      "paper_title": "...",
      "num_tables": N,
      "tables": [
        {
          "table_id": "table_1",
          "page": X,
          "evaluation": { "expected_rows": N, "expected_cols": M, "columns": [...] },
          "rows": [...]
        }
      ]
    }
  ]
}
```

In [41]:
import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path
from bs4 import BeautifulSoup
import deepdoctection as dd

## Configuration


In [42]:
# Directorio con los PDFs y ruta de salida
PDF_DIR = Path("pdfs_prueba")
OUTPUT_PATH = Path("pdfs_prueba/ground_truth/ground_truth_deepdoctection.json")

# Los 5 papers a procesar
PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))
print(f"PDFs encontrados: {len(PDF_FILES)}")
for p in PDF_FILES:
    print(f"  - {p.name}")

PDFs encontrados: 5
  - Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function.pdf
  - Adversarial Contrastive Estimation.pdf
  - Binarized Knowledge Graph Embeddings.pdf
  - HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge Base Completion.pdf
  - Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text.pdf


## Funciones auxiliares

In [43]:
def coerce_value(s: str):
    """Convierte un string a número si es posible. Valores vacíos/faltantes → '-'."""
    s = s.strip()
    if not s or s in ('–', 'N/A', 'n/a', 'nan'):
        return '-'
    if s == '-':
        return '-'
    s_clean = s.replace(',', '.')
    try:
        val = float(s_clean)
        # Devolver entero si no tiene parte decimal significativa
        return int(val) if val == int(val) else val
    except ValueError:
        return s


def build_grid(rows_elements):
    """
    Construye una cuadrícula 2D resolviendo rowspan/colspan.
    Devuelve: grid dict {(row, col): text}, max_row, max_col, max_header_reach
    """
    grid = {}
    max_header_reach = 1  # máximo alcance de spans en filas de datos

    for row_idx, row in enumerate(rows_elements):
        col_idx = 0
        for cell in row.find_all(['td', 'th']):
            # Saltar celdas ya ocupadas por un span previo
            while (row_idx, col_idx) in grid:
                col_idx += 1

            content = cell.get_text(separator=' ', strip=True)
            rowspan = int(cell.get('rowspan', 1))
            colspan = int(cell.get('colspan', 1))

            if rowspan > 1 or colspan > 1:
                max_header_reach = max(max_header_reach, row_idx + rowspan)

            for dr in range(rowspan):
                for dc in range(colspan):
                    grid[(row_idx + dr, col_idx + dc)] = content

            col_idx += colspan

    if not grid:
        return {}, 0, 0, 1

    max_row = max(r for r, _ in grid.keys()) + 1
    max_col = max(c for _, c in grid.keys()) + 1
    return grid, max_row, max_col, max_header_reach


def flatten_headers(grid, header_row_count: int, max_col: int):
    """
    Aplana cabeceras multinivel combinando con '_'.
    Elimina valores repetidos consecutivos (causados por rowspan propagation).
    Ejemplo: ['Models', 'WN18_MRR', 'WN18_Hits@_1', 'WN18_Hits@_3', ...]
    """
    columns = []
    for col_idx in range(max_col):
        parts = []
        prev = None
        for row_idx in range(header_row_count):
            val = grid.get((row_idx, col_idx), '').strip()
            if val and val != prev:
                parts.append(val)
            prev = val
        columns.append('_'.join(parts) if parts else '')
    return columns


def parse_table_html(html_str: str):
    """
    Parsea el HTML de una tabla extraída por deepdoctection.

    - Resuelve rowspan/colspan para detectar cuántas filas son cabecera.
    - Aplana encabezados multinivel (convención Dataset_Metrica_Variante).
    - Convierte valores numéricos a int/float; '-' para faltantes.

    Devuelve: (columns: list, data_rows: list[list]) o (None, None) si falla.
    """
    if not html_str:
        return None, None

    soup = BeautifulSoup(html_str, 'html.parser')
    table = soup.find('table')
    if not table:
        return None, None

    rows_elements = table.find_all('tr')
    if not rows_elements:
        return None, None

    grid, max_row, max_col, max_header_reach = build_grid(rows_elements)

    if not grid or max_col == 0 or max_row == 0:
        return None, None

    # Las filas de cabecera son las que forman el bloque de spans iniciales.
    # Debe quedar al menos 1 fila de datos.
    header_row_count = max(1, max_header_reach)
    header_row_count = min(header_row_count, max_row - 1)

    columns = flatten_headers(grid, header_row_count, max_col)

    # Extraer filas de datos
    data_rows = []
    for row_idx in range(header_row_count, max_row):
        row = [coerce_value(grid.get((row_idx, c), '')) for c in range(max_col)]
        # Ignorar filas completamente vacías o de guiones
        if all(v in ('-', '') for v in row):
            continue
        data_rows.append(row)

    return columns, data_rows


print("Funciones auxiliares cargadas.")

Funciones auxiliares cargadas.


## PDF text extraction (PDF Miner)


In [44]:
analyzer = dd.get_dd_analyzer(config_overwrite=[
    'USE_PDF_MINER=True',
    'USE_OCR=False',
    'OCR.USE_DOCTR=False',
])
print("Analyzer listo:", type(analyzer).__name__)

[0407 13:00.14 @dd.py:119]  INF  Config: 
 {'CELL': {'FILTER': None,
          'PAD': {'BOTTOM': 60, 'LEFT': 60, 'RIGHT': 60, 'TOP': 60},
          'PADDING': False,
          'WEIGHTS': 'cell/d2_model_1849999_cell_inf_only.pt',
          'WEIGHTS_TS': 'cell/d2_model_1849999_cell_inf_only.ts'},
 'DEVICE': device(type='mps'),
 'ENFORCE_WEIGHTS': {'CELL': True, 'ITEM': True, 'LAYOUT': True},
 'ITEM': {'FILTER': ['table'],
          'PAD': {'BOTTOM': 60, 'LEFT': 60, 'RIGHT': 60, 'TOP': 60},
          'PADDING': False,
          'WEIGHTS': 'deepdoctection/tatr_tab_struct_v2/model.safetensors',
          'WEIGHTS_TS': 'item/d2_model_1639999_item_inf_only.ts'},
 'LANGUAGE': None,
 'LAYOUT': {'FILTER': None,
            'PAD': {'BOTTOM': 0, 'LEFT': 0, 'RIGHT': 0, 'TOP': 0},
            'PADDING': False,
            'WEIGHTS': 'Aryn/deformable-detr-DocLayNet/model.safetensors',
            'WEIGHTS_TS': 'layout/d2_model_0829999_layout_inf_only.ts'},
 'LAYOUT_LINK': {'CHILD_CATEGORIES': [<Layou

Analyzer listo: DoctectionPipe


## Extracción de tablas

In [45]:
def extract_tables_from_pdf(pdf_path: Path, analyzer) -> dict:
    """
    Procesa un PDF con deepdoctection y devuelve un documento con el
    formato del ground truth (paper_title, num_tables, tables[]).
    """
    paper_title = pdf_path.stem
    print(f"\n{'='*70}")
    print(f"Procesando: {paper_title}")
    print(f"{'='*70}")

    df = analyzer.analyze(path=str(pdf_path))
    df.reset_state()

    # Recolectar todas las tablas en orden de aparición (página, luego orden en página)
    all_tables_raw = []
    for page in df:
        for table in page.tables:
            all_tables_raw.append((page.page_number, table))

    print(f"Tablas detectadas: {len(all_tables_raw)}")

    tables_json = []
    table_counter = 0

    for page_num, table in all_tables_raw:
        html_str = table.html

        if not html_str:
            print(f"  [SKIP] página {page_num}: sin HTML")
            continue

        columns, data_rows = parse_table_html(html_str)

        if columns is None or not data_rows:
            print(f"  [SKIP] página {page_num}: no parseada o sin filas de datos")
            continue

        table_counter += 1
        table_id = f"table_{table_counter}"
        expected_rows = len(data_rows)
        expected_cols = len(columns)

        print(f"  [{table_id}] página {page_num} → {expected_rows} filas × {expected_cols} cols")
        print(f"    Columnas: {columns}")

        tables_json.append({
            "table_id": table_id,
            "page": page_num,
            "evaluation": {
                "expected_rows": expected_rows,
                "expected_cols": expected_cols,
                "columns": columns
            },
            "rows": data_rows
        })

    document = {
        "paper_title": paper_title,
        "num_tables": len(tables_json),
        "tables": tables_json
    }

    print(f"  → Tablas incluidas en ground truth: {len(tables_json)}")
    return document

## Ejecutar en los 5 papers

In [46]:
documents = []

for pdf_path in PDF_FILES:
    doc = extract_tables_from_pdf(pdf_path, analyzer)
    documents.append(doc)

ground_truth = {"documents": documents}

print(f"\n{'='*70}")
print("RESUMEN FINAL")
print(f"{'='*70}")
total_tables = sum(d['num_tables'] for d in documents)
for d in documents:
    print(f"  {d['paper_title'][:60]:60s} → {d['num_tables']} tablas")
print(f"\nTotal tablas extraídas: {total_tables}")

[0407 13:00.15 @doctectionpipe.py:118]  INF  Processing Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function_0.pdf



Procesando: Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function


[0407 13:00.30 @context.py:156]  INF  ImageLayoutService total: 14.8391 sec.
[0407 13:00.30 @context.py:156]  INF  AnnotationNmsService total: 0.0019 sec.
[0407 13:00.30 @context.py:156]  INF  SubImageLayoutService total: 0.0001 sec.
[0407 13:00.30 @context.py:156]  INF  PubtablesSegmentationService total: 0.0004 sec.
[0407 13:00.30 @context.py:156]  INF  TextExtractionService total: 0.1303 sec.
[0407 13:00.30 @context.py:156]  INF  MatchingService total: 0.0046 sec.
[0407 13:00.30 @context.py:156]  INF  TextOrderService total: 0.0098 sec.
[0407 13:00.30 @doctectionpipe.py:118]  INF  Processing Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function_1.pdf
[0407 13:00.32 @context.py:156]  INF  ImageLayoutService total: 1.8256 sec.
[0407 13:00.32 @context.py:156]  INF  AnnotationNmsService total: 0.0009 sec.
[0407 13:00.32 @context.py:156]  INF  SubImageLayoutService total: 0.0 sec.
[0407 13:00.32 @context.py:156]  INF  PubtablesSegmentationServic

Tablas detectadas: 2
  [table_1] página 4 → 7 filas × 3 cols
    Columnas: ['Data', 'FB15k', 'wn18']
  [table_2] página 4 → 18 filas × 9 cols
    Columnas: ['Dataset', 'WN18_Mean_raw', 'WN18_Mean_filter', 'WN18_Hits@10(%)_raw', 'WN18_Hits@10(%)_filter', 'FB15k_Mean_raw', 'FB15k_Mean_filter', 'FB15k_Hits@10(%)_raw', 'FB15k_Hits@10(%)_filter']
  → Tablas incluidas en ground truth: 2

Procesando: Adversarial Contrastive Estimation


[0407 13:00.48 @doctectionpipe.py:118]  INF  Processing Adversarial Contrastive Estimation_0.pdf
[0407 13:00.54 @context.py:156]  INF  ImageLayoutService total: 5.8708 sec.
[0407 13:00.54 @context.py:156]  INF  AnnotationNmsService total: 0.0029 sec.
[0407 13:00.54 @context.py:156]  INF  SubImageLayoutService total: 0.0001 sec.
[0407 13:00.54 @context.py:156]  INF  PubtablesSegmentationService total: 0.0001 sec.
[0407 13:00.55 @context.py:156]  INF  TextExtractionService total: 0.6357 sec.
[0407 13:00.55 @context.py:156]  INF  MatchingService total: 0.0115 sec.
[0407 13:00.55 @context.py:156]  INF  TextOrderService total: 0.0222 sec.
[0407 13:00.55 @doctectionpipe.py:118]  INF  Processing Adversarial Contrastive Estimation_1.pdf
[0407 13:00.58 @context.py:156]  INF  ImageLayoutService total: 3.3577 sec.
[0407 13:00.58 @context.py:156]  INF  AnnotationNmsService total: 0.0054 sec.
[0407 13:00.58 @context.py:156]  INF  SubImageLayoutService total: 0.0001 sec.
[0407 13:00.58 @context.py:1

...

  [table_4] página 9 → 10 filas × 5 cols
    Columnas: ['Model_DistMult*(D=200) 6,400 ComplEx*(D=200) 12,800_ConvE*(D=200)_CP(D=15) CP(D=50)', 'Bitsperentity_DistMult*(D=200) 6,400 ComplEx*(D=200) 12,800_6,400_960_3,200', 'Bitsperrelation_6,400_12,800_6,400_480_1,600', 'MRR_WN18RR_43.0_44.0_43.0_40.0_43.0', 'MRR_FB15k-237_24.1_24.7_32.5_22.0_24.8']
  [table_5] página 11 → 3 filas × 3 cols
    Columnas: ['', 'Accuracy', 'Modelsize']
  → Tablas incluidas en ground truth: 5

Procesando: HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge Base Completion


[0407 13:02.02 @context.py:156]  INF  ImageLayoutService total: 1.7683 sec.
[0407 13:02.02 @context.py:156]  INF  AnnotationNmsService total: 0.0017 sec.
[0407 13:02.02 @context.py:156]  INF  SubImageLayoutService total: 0.0001 sec.
[0407 13:02.02 @context.py:156]  INF  PubtablesSegmentationService total: 0.0001 sec.
[0407 13:02.02 @context.py:156]  INF  TextExtractionService total: 0.1247 sec.
[0407 13:02.02 @context.py:156]  INF  MatchingService total: 0.0052 sec.
[0407 13:02.02 @context.py:156]  INF  TextOrderService total: 0.0102 sec.
[0407 13:02.02 @doctectionpipe.py:118]  INF  Processing HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge Base Completion_1.pdf
[0407 13:02.04 @context.py:156]  INF  ImageLayoutService total: 1.8186 sec.
[0407 13:02.04 @context.py:156]  INF  AnnotationNmsService total: 0.0011 sec.
[0407 13:02.04 @context.py:156]  INF  SubImageLayoutService total: 0.0 sec.
[0407 13:02.04 @context.py:156]  INF  PubtablesSegmentationService total: 0.0001 sec.


Tablas detectadas: 5
  [table_1] página 5 → 4 filas × 6 cols
    Columnas: ['Dataset', '|E|', '|R|', '#Train', '#Valid', '#Test']
  [table_2] página 5 → 1 filas × 3 cols
    Columnas: ['(a)', 'is a(x,y)∧part', 'of(y,z)→part of(x,z)']
  [table_3] página 6 → 6 filas × 6 cols
    Columnas: ['Method', 'Type', 'WN18RR_MRR', 'WN18RR_H@10', 'FB15k-237_MRR', 'FB15k-237_H@10']
  [table_4] página 6 → 2 filas × 5 cols
    Columnas: ['Method', 'MRR', 'WD H@10', 'WD MRR', '++ H@10']
  [table_5] página 8 → 8 filas × 9 cols
    Columnas: ['Dataset', 'Model', '# negsE', '# negsR', 'η', 'λ', 'n', 'γ,', 'β']
  → Tablas incluidas en ground truth: 5

Procesando: Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text


[0407 13:02.25 @context.py:156]  INF  ImageLayoutService total: 1.782 sec.
[0407 13:02.25 @context.py:156]  INF  AnnotationNmsService total: 0.0011 sec.
[0407 13:02.25 @context.py:156]  INF  SubImageLayoutService total: 0.0001 sec.
[0407 13:02.25 @context.py:156]  INF  PubtablesSegmentationService total: 0.0001 sec.
[0407 13:02.25 @context.py:156]  INF  TextExtractionService total: 0.149 sec.
[0407 13:02.25 @context.py:156]  INF  MatchingService total: 0.0063 sec.
[0407 13:02.25 @context.py:156]  INF  TextOrderService total: 0.0158 sec.
[0407 13:02.25 @doctectionpipe.py:118]  INF  Processing Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text_1.pdf
[0407 13:02.27 @context.py:156]  INF  ImageLayoutService total: 1.7464 sec.
[0407 13:02.27 @context.py:156]  INF  AnnotationNmsService total: 0.0009 sec.
[0407 13:02.27 @context.py:156]  INF  SubImageLayoutService total: 0.0 sec.
[0407 13:02.27 @context.py:156]  INF  PubtablesSegmentationService total: 0.0 sec.

Tablas detectadas: 2
  [table_1] página 2 → 1 filas × 2 cols
    Columnas: ['Text', 'Berlin is the capital city of Germany.']
  [table_2] página 3 → 5 filas × 4 cols
    Columnas: ['Tasks Metric', 'NYT F1-MeasureF1-Measure', 'ADE', 'Wiki-DBpedia F1-Measure']
  → Tablas incluidas en ground truth: 2

RESUMEN FINAL
  Adaptive Margin Ranking Loss for Knowledge Graph Embeddings  → 2 tablas
  Adversarial Contrastive Estimation                           → 4 tablas
  Binarized Knowledge Graph Embeddings                         → 5 tablas
  HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge → 5 tablas
  Seq2RDF- An end-to-end application for deriving Triples from → 2 tablas

Total tablas extraídas: 18


## Compare table counts to manual ground truth


In [47]:
gt_manual_path = Path("pdfs_prueba/ground_truth/ground_truth_kge.json")

if gt_manual_path.exists():
    with open(gt_manual_path, 'r', encoding='utf-8') as f:
        gt_manual = json.load(f)
    manual_by_title = {d['paper_title']: d for d in gt_manual['documents']}

    print(f"{'Paper':<55} {'Manual':>8} {'DD':>6} {'Match':>6}")
    print('-' * 80)
    for doc in documents:
        title = doc['paper_title']
        dd_count = doc['num_tables']
        manual_doc = manual_by_title.get(title)
        manual_count = manual_doc['num_tables'] if manual_doc else '??'
        match = 'OK' if manual_doc and dd_count == manual_doc['num_tables'] else 'DIFF'
        print(f"{title[:55]:<55} {str(manual_count):>8} {dd_count:>6} {match:>6}")
else:
    print("Ground truth manual no encontrado.")

Paper                                                     Manual     DD  Match
--------------------------------------------------------------------------------
Adaptive Margin Ranking Loss for Knowledge Graph Embedd        2      2     OK
Adversarial Contrastive Estimation                             4      4     OK
Binarized Knowledge Graph Embeddings                           5      5     OK
HyperKG- Hyperbolic Knowledge Graph Embeddings for Know        4      5   DIFF
Seq2RDF- An end-to-end application for deriving Triples        2      2     OK


## Per-paper table preview


In [48]:
import pandas as pd

for doc in documents:
    print(f"\n{'='*70}")
    print(f"  {doc['paper_title']}")
    if not doc['tables']:
        print("  (sin tablas)")
        continue
    for t in doc['tables']:
        print(f"\n  [{t['table_id']}] página {t['page']} — "
              f"{t['evaluation']['expected_rows']} filas × {t['evaluation']['expected_cols']} cols")
        df_preview = pd.DataFrame(t['rows'], columns=t['evaluation']['columns'])
        display(df_preview.head(5))  # mostrar sólo las 5 primeras filas


  Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function

  [table_1] página 4 — 7 filas × 3 cols


,Data,FB15k,wn18
0,Optimizer,Adagrad,Adagrad
1,Embeddingsize,100,100
2,Epochs ξ,900000 0.1,900000 0.1
3,σ,1,1
4,λ,1,1



  [table_2] página 4 — 18 filas × 9 cols


,Dataset,WN18_Mean_raw,WN18_Mean_filter,WN18_Hits@10(%)_raw,WN18_Hits@10(%)_filter,FB15k_Mean_raw,FB15k_Mean_filter,FB15k_Hits@10(%)_raw,FB15k_Hits@10(%)_filter
0,Unstructured[6],315,304,35.5,38.2,1074,979,4.5,6.3
1,RESCALNickeletal.[22],1180,1163,37.2,52.8,828,683,28.4,44.1
2,SE[8],1011,985,68.5,80.5,273,162,28.8,39.8
3,SME(linear)[6],545,533,65.1,74.1,274,154,30.7,40.8
4,SME(bilinear)[6],526,509,54.7,61.3,284,158,31.3,41.3



  Adversarial Contrastive Estimation

  [table_1] página 7 — 9 filas × 3 cols


...

,Dataset,Model,# negsE,# negsR,η,λ,n,"γ,",β
0,WN18RR,HyperKG,10,0,0.01,0.8,100,1.0,(cid:98)n(cid:99) 2
1,WN18RR,HyperKG(Mo¨biusaddition),10,0,0.01,-,100,1.0,(cid:98)n(cid:99) 2
2,WN18RR,HyperKG(noregularisation),10,0,0.01,0,100,1.0,(cid:98)n(cid:99) 2
3,FB15k-237,HyperKG,5,0,0.01,0.2,100,0.5,(cid:98)n(cid:99) 2
4,FB15k-237,HyperKG(Mo¨biusaddition),5,0,0.01,-,100,0.5,(cid:98)n(cid:99) 2



  Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text

  [table_1] página 2 — 1 filas × 2 cols


,Text,Berlin is the capital city of Germany.
0,Triple,dbr:Germany dbo:capital dbr:Berlin



  [table_2] página 3 — 5 filas × 4 cols


,Tasks Metric,NYT F1-MeasureF1-Measure,ADE,Wiki-DBpedia F1-Measure
0,EL+Lexical,36.8,61.4,37.8
1,EL+LSTM,58.7,70.3,65.5
2,EL+GRU,59.8,73.2,67.0
3,Seq2Seq,64.2,73.4,73.5
4,S+A+W+G,71.4,79.5,84.3


## Comparación detallada tabla a tabla

Para cada tabla, compara filas, columnas y estructura con el ground truth manual.  
Las diferencias marcadas con `DIFF` son candidatas a revisión manual.

In [49]:
if gt_manual_path.exists():
    for doc in documents:
        title = doc['paper_title']
        manual_doc = manual_by_title.get(title)
        if not manual_doc:
            print(f"\n[{title}] No encontrado en ground truth manual")
            continue

        print(f"\n{'='*70}")
        print(f"Paper: {title}")
        print(f"  Tablas manual: {manual_doc['num_tables']}  |  DD: {doc['num_tables']}")

        for i, dd_table in enumerate(doc['tables']):
            row_match = col_match = '?'
            m_rows = m_cols = '?'

            if i < len(manual_doc['tables']):
                m_table = manual_doc['tables'][i]
                m_rows = m_table['evaluation']['expected_rows']
                m_cols = m_table['evaluation']['expected_cols']
                row_match = 'OK' if dd_table['evaluation']['expected_rows'] == m_rows else 'DIFF'
                col_match = 'OK' if dd_table['evaluation']['expected_cols'] == m_cols else 'DIFF'

            print(f"\n  [{dd_table['table_id']}] página {dd_table['page']}")
            print(f"    DD     → {dd_table['evaluation']['expected_rows']} filas × {dd_table['evaluation']['expected_cols']} cols")
            print(f"    Manual → {m_rows} filas × {m_cols} cols")
            print(f"    Filas {row_match}  |  Columnas {col_match}")

            if col_match == 'DIFF':
                print(f"    Col DD:     {dd_table['evaluation']['columns']}")
                if i < len(manual_doc['tables']):
                    print(f"    Col Manual: {manual_doc['tables'][i]['evaluation']['columns']}")
else:
    print("Ground truth manual no disponible.")


Paper: Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function
  Tablas manual: 2  |  DD: 2

  [table_1] página 4
    DD     → 7 filas × 3 cols
    Manual → 19 filas × 9 cols
    Filas DIFF  |  Columnas DIFF
    Col DD:     ['Data', 'FB15k', 'wn18']
    Col Manual: ['Dataset', 'WN18_Mean_raw', 'WN18_Mean_filter', 'WN18_Hits@10_raw', 'WN18_Hits@10_filter', 'FB15k_Mean_raw', 'FB15k_Mean_filter', 'FB15k_Hits@10_raw', 'FB15k_Hits@10_filter']

  [table_2] página 4
    DD     → 18 filas × 9 cols
    Manual → 8 filas × 3 cols
    Filas DIFF  |  Columnas DIFF
    Col DD:     ['Dataset', 'WN18_Mean_raw', 'WN18_Mean_filter', 'WN18_Hits@10(%)_raw', 'WN18_Hits@10(%)_filter', 'FB15k_Mean_raw', 'FB15k_Mean_filter', 'FB15k_Hits@10(%)_raw', 'FB15k_Hits@10(%)_filter']
    Col Manual: ['Data', 'FB15k', 'wn18']

Paper: Adversarial Contrastive Estimation
  Tablas manual: 4  |  DD: 4

  [table_1] página 7
    DD     → 9 filas × 3 cols
    Manual → 20 filas × 6 cols

## Guardar ground truth generado

Guarda el resultado en `ground_truth_deepdoctection.json` para revisión manual posterior.

In [50]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(ground_truth, f, indent=2, ensure_ascii=False)

print(f"Ground truth guardado en: {OUTPUT_PATH}")
print(f"Documentos:    {len(documents)}")
print(f"Total tablas:  {sum(d['num_tables'] for d in documents)}")

Ground truth guardado en: pdfs_prueba/ground_truth/ground_truth_deepdoctection.json
Documentos:    5
Total tablas:  18


# Generación de Ground Truth con DeepDoctection

Este notebook extrae tablas de los PDFs usando **deepdoctection** y genera un archivo JSON con el mismo formato del ground truth manual (`ground_truth_kge.json`).

El resultado será revisado y ajustado manualmente según las instrucciones en `ground_truth_manual.md`.

**Formato de salida:**
```json
{
  "documents": [
    {
      "paper_title": "...",
      "num_tables": N,
      "tables": [
        {
          "table_id": "table_1",
          "page": X,
          "evaluation": { "expected_rows": N, "expected_cols": M, "columns": [...] },
          "rows": [...]
        }
      ]
    }
  ]
}
```

In [51]:
import warnings
warnings.filterwarnings('ignore')

import json
import re
from pathlib import Path
from bs4 import BeautifulSoup
import deepdoctection as dd

## Configuración

In [52]:
# Directorio con los PDFs y ruta de salida
PDF_DIR = Path("pdfs_prueba")
OUTPUT_PATH = Path("pdfs_prueba/ground_truth/ground_truth_deepdoctection.json")

# Los 5 papers a procesar
PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))
print(f"PDFs encontrados: {len(PDF_FILES)}")
for p in PDF_FILES:
    print(f"  - {p.name}")

PDFs encontrados: 5
  - Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function.pdf
  - Adversarial Contrastive Estimation.pdf
  - Binarized Knowledge Graph Embeddings.pdf
  - HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge Base Completion.pdf
  - Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text.pdf


## Funciones auxiliares

In [53]:
import re


def coerce_value(s: str):
    """
    Convierte un string a número si es posible; si es vacío/faltante, devuelve '-'.
    Maneja correctamente separadores de miles (ej. '40,943' → 40943, no 40.943).
    """
    s = s.strip()
    if not s or s in ('–', 'N/A', 'n/a', 'nan'):
        return '-'
    if s == '-':
        return '-'

    # Detectar si usa coma como separador de miles (ej. '40,943', '272,115')
    # vs. coma como separador decimal (ej. '0,83')
    # Heurística: si hay una coma seguida de exactamente 3 dígitos al final → miles
    s_clean = s
    if re.search(r',\d{3}(\D|$)', s):
        # Separador de miles: eliminar comas
        s_clean = s.replace(',', '')
    else:
        # Posible decimal europeo: sustituir coma por punto
        s_clean = s.replace(',', '.')

    try:
        val = float(s_clean)
        return int(val) if val == int(val) else val
    except ValueError:
        return s


def build_grid(rows_elements):
    """
    Construye una cuadrícula 2D resolviendo rowspan/colspan.
    Devuelve: grid dict {(row, col): text}, max_row, max_col, max_rowspan_reach,
              header_rows set (índices de filas que contienen <th>)
    """
    grid = {}
    max_rowspan_reach = 1
    th_rows = set()  # filas que contienen al menos un <th>

    for row_idx, row in enumerate(rows_elements):
        col_idx = 0
        for cell in row.find_all(['td', 'th']):
            while (row_idx, col_idx) in grid:
                col_idx += 1

            content = cell.get_text(separator=' ', strip=True)
            rowspan = int(cell.get('rowspan', 1))
            colspan = int(cell.get('colspan', 1))

            if cell.name == 'th':
                th_rows.add(row_idx)

            if rowspan > 1 or colspan > 1:
                max_rowspan_reach = max(max_rowspan_reach, row_idx + rowspan)

            for dr in range(rowspan):
                for dc in range(colspan):
                    grid[(row_idx + dr, col_idx + dc)] = content

            col_idx += colspan

    if not grid:
        return {}, 0, 0, 1, set()

    max_row = max(r for r, _ in grid.keys()) + 1
    max_col = max(c for _, c in grid.keys()) + 1
    return grid, max_row, max_col, max_rowspan_reach, th_rows


def detect_header_rows(th_rows, max_rowspan_reach, max_row):
    """
    Determina cuántas filas son de cabecera usando dos señales:
      1. Etiquetas <th> en el HTML (más fiable que contar spans).
      2. El alcance máximo de rowspan/colspan.
    Garantiza que siempre queda al menos 1 fila de datos.
    """
    if th_rows:
        # Tomar el rango continuo de filas <th> desde la fila 0
        max_th_row = max(th_rows)
        # Solo considerar filas <th> contiguas desde el inicio
        header_count = 0
        for i in range(max_th_row + 1):
            if i in th_rows or i <= max_th_row:
                header_count = i + 1
    else:
        # Fallback: usar alcance de spans
        header_count = max(1, max_rowspan_reach)

    # Nunca consumir todas las filas como cabecera
    return min(header_count, max_row - 1)


def flatten_headers(grid, header_row_count, max_col):
    """
    Aplana cabeceras multinivel para cada columna.
    - Elimina repeticiones consecutivas (causadas por rowspan propagation).
    - Corrige @_N → @N (underscore espurio antes de número tras @).
    Resultado: ['Models', 'WN18_MRR', 'WN18_Hits@1', 'WN18_Hits@10', ...]
    """
    columns = []
    for col_idx in range(max_col):
        parts = []
        prev = None
        for row_idx in range(header_row_count):
            val = grid.get((row_idx, col_idx), '').strip()
            if val and val != prev:
                parts.append(val)
            prev = val
        col_name = '_'.join(parts) if parts else ''
        # Corregir underscore espurio antes de número después de '@' → 'Hits@_10' → 'Hits@10'
        col_name = re.sub(r'@_(\d)', r'@\1', col_name)
        columns.append(col_name)
    return columns


def parse_table_html(html_str: str):
    """
    Parsea el HTML de una tabla de deepdoctection.
    - Detecta cabeceras usando etiquetas <th> (señal primaria) o spans (fallback).
    - Aplana encabezados multinivel y corrige artefactos tipográficos.
    - Convierte valores numéricos (manejo correcto de separadores de miles).

    Devuelve: (columns: list, data_rows: list[list]) o (None, None) si falla.
    """
    if not html_str:
        return None, None

    soup = BeautifulSoup(html_str, 'html.parser')
    table = soup.find('table')
    if not table:
        return None, None

    rows_elements = table.find_all('tr')
    if not rows_elements:
        return None, None

    grid, max_row, max_col, max_rowspan_reach, th_rows = build_grid(rows_elements)

    if not grid or max_col == 0 or max_row < 2:
        return None, None

    header_row_count = detect_header_rows(th_rows, max_rowspan_reach, max_row)

    columns = flatten_headers(grid, header_row_count, max_col)

    # Extraer filas de datos
    data_rows = []
    for row_idx in range(header_row_count, max_row):
        row = [coerce_value(grid.get((row_idx, c), '')) for c in range(max_col)]
        if all(v in ('-', '') for v in row):
            continue
        data_rows.append(row)

    return columns, data_rows


print("Funciones auxiliares cargadas.")


Funciones auxiliares cargadas.


## Inicializar DeepDoctection

Se usa **PDF Miner** (sin OCR) ya que los PDFs son texto nativo, lo que es más rápido y preciso.

In [54]:
analyzer = dd.get_dd_analyzer(config_overwrite=[
    'USE_PDF_MINER=True',
    'USE_OCR=False',
    'OCR.USE_DOCTR=False',
])
print("Analyzer listo:", type(analyzer).__name__)

[0407 13:02.33 @dd.py:119]  INF  Config: 
 {'CELL': {'FILTER': None,
          'PAD': {'BOTTOM': 60, 'LEFT': 60, 'RIGHT': 60, 'TOP': 60},
          'PADDING': False,
          'WEIGHTS': 'cell/d2_model_1849999_cell_inf_only.pt',
          'WEIGHTS_TS': 'cell/d2_model_1849999_cell_inf_only.ts'},
 'DEVICE': device(type='mps'),
 'ENFORCE_WEIGHTS': {'CELL': True, 'ITEM': True, 'LAYOUT': True},
 'ITEM': {'FILTER': ['table'],
          'PAD': {'BOTTOM': 60, 'LEFT': 60, 'RIGHT': 60, 'TOP': 60},
          'PADDING': False,
          'WEIGHTS': 'deepdoctection/tatr_tab_struct_v2/model.safetensors',
          'WEIGHTS_TS': 'item/d2_model_1639999_item_inf_only.ts'},
 'LANGUAGE': None,
 'LAYOUT': {'FILTER': None,
            'PAD': {'BOTTOM': 0, 'LEFT': 0, 'RIGHT': 0, 'TOP': 0},
            'PADDING': False,
            'WEIGHTS': 'Aryn/deformable-detr-DocLayNet/model.safetensors',
            'WEIGHTS_TS': 'layout/d2_model_0829999_layout_inf_only.ts'},
 'LAYOUT_LINK': {'CHILD_CATEGORIES': [<Layou

Analyzer listo: DoctectionPipe


## Extracción de tablas por paper

In [55]:
def extract_tables_from_pdf(pdf_path: Path, analyzer) -> dict:
    """
    Procesa un PDF con deepdoctection y devuelve una estructura de documento
    con el formato del ground truth.
    Nota: deepdoctection numera páginas desde 0; se suma +1 para coincidir
    con el GT manual (numeración 1-indexada).
    """
    paper_title = pdf_path.stem
    print(f"\n{'='*70}")
    print(f"Procesando: {paper_title}")
    print(f"{'='*70}")

    df = analyzer.analyze(path=str(pdf_path))
    df.reset_state()

    # Recolectar tablas de todas las páginas en orden de aparición
    all_tables_raw = []
    for page in df:
        # page.page_number es 0-indexed → +1 para numeración 1-indexada
        page_num_1based = page.page_number + 1
        for table in page.tables:
            all_tables_raw.append((page_num_1based, table))

    print(f"Tablas detectadas por deepdoctection: {len(all_tables_raw)}")

    tables_json = []
    table_counter = 0

    for page_num, table in all_tables_raw:
        html_str = table.html

        if not html_str:
            print(f"  [SKIP] tabla en página {page_num}: sin HTML")
            continue

        columns, data_rows = parse_table_html(html_str)

        if columns is None or not data_rows:
            print(f"  [SKIP] tabla en página {page_num}: no se pudo parsear o sin filas de datos")
            continue

        table_counter += 1
        table_id = f"table_{table_counter}"
        expected_rows = len(data_rows)
        expected_cols = len(columns)

        print(f"  [{table_id}] página {page_num} → {expected_rows} filas × {expected_cols} cols")
        print(f"    Columnas: {columns}")

        tables_json.append({
            "table_id": table_id,
            "page": page_num,
            "evaluation": {
                "expected_rows": expected_rows,
                "expected_cols": expected_cols,
                "columns": columns
            },
            "rows": data_rows
        })

    document = {
        "paper_title": paper_title,
        "num_tables": len(tables_json),
        "tables": tables_json
    }

    print(f"  → Tablas incluidas en ground truth: {len(tables_json)}")
    return document


## Ejecutar extracción en los 5 papers

In [56]:
documents = []

for pdf_path in PDF_FILES:
    doc = extract_tables_from_pdf(pdf_path, analyzer)
    documents.append(doc)

ground_truth = {"documents": documents}

print(f"\n{'='*70}")
print("RESUMEN FINAL")
print(f"{'='*70}")
total_tables = sum(d['num_tables'] for d in documents)
for d in documents:
    print(f"  {d['paper_title'][:60]:60s} → {d['num_tables']} tablas")
print(f"\nTotal tablas extraídas: {total_tables}")


Procesando: Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function


[0407 13:02.34 @doctectionpipe.py:118]  INF  Processing Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function_0.pdf
[0407 13:02.37 @context.py:156]  INF  ImageLayoutService total: 2.8548 sec.
[0407 13:02.37 @context.py:156]  INF  AnnotationNmsService total: 0.0018 sec.
[0407 13:02.37 @context.py:156]  INF  SubImageLayoutService total: 0.0001 sec.
[0407 13:02.37 @context.py:156]  INF  PubtablesSegmentationService total: 0.0002 sec.
[0407 13:02.37 @context.py:156]  INF  TextExtractionService total: 0.131 sec.
[0407 13:02.37 @context.py:156]  INF  MatchingService total: 0.0042 sec.
[0407 13:02.37 @context.py:156]  INF  TextOrderService total: 0.0093 sec.
[0407 13:02.37 @doctectionpipe.py:118]  INF  Processing Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function_1.pdf
[0407 13:02.39 @context.py:156]  INF  ImageLayoutService total: 1.8022 sec.
[0407 13:02.39 @context.py:156]  INF  AnnotationNmsService tot

Tablas detectadas por deepdoctection: 2
  [table_1] página 5 → 7 filas × 3 cols
    Columnas: ['Data', 'FB15k', 'wn18']


[0407 13:02.51 @doctectionpipe.py:118]  INF  Processing Adversarial Contrastive Estimation_0.pdf


  [table_2] página 5 → 18 filas × 9 cols
    Columnas: ['Dataset', 'WN18_Mean_raw', 'WN18_Mean_filter', 'WN18_Hits@10(%)_raw', 'WN18_Hits@10(%)_filter', 'FB15k_Mean_raw', 'FB15k_Mean_filter', 'FB15k_Hits@10(%)_raw', 'FB15k_Hits@10(%)_filter']
  → Tablas incluidas en ground truth: 2

Procesando: Adversarial Contrastive Estimation


...

  [table_4] página 10 → 10 filas × 5 cols
    Columnas: ['Model_DistMult*(D=200) 6,400 ComplEx*(D=200) 12,800_ConvE*(D=200)_CP(D=15) CP(D=50)', 'Bitsperentity_DistMult*(D=200) 6,400 ComplEx*(D=200) 12,800_6,400_960_3,200', 'Bitsperrelation_6,400_12,800_6,400_480_1,600', 'MRR_WN18RR_43.0_44.0_43.0_40.0_43.0', 'MRR_FB15k-237_24.1_24.7_32.5_22.0_24.8']
  [table_5] página 12 → 3 filas × 3 cols
    Columnas: ['', 'Accuracy', 'Modelsize']
  → Tablas incluidas en ground truth: 5

Procesando: HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge Base Completion


[0407 13:03.54 @context.py:156]  INF  ImageLayoutService total: 1.7734 sec.
[0407 13:03.54 @context.py:156]  INF  AnnotationNmsService total: 0.0015 sec.
[0407 13:03.54 @context.py:156]  INF  SubImageLayoutService total: 0.0001 sec.
[0407 13:03.54 @context.py:156]  INF  PubtablesSegmentationService total: 0.0003 sec.
[0407 13:03.55 @context.py:156]  INF  TextExtractionService total: 0.1259 sec.
[0407 13:03.55 @context.py:156]  INF  MatchingService total: 0.0054 sec.
[0407 13:03.55 @context.py:156]  INF  TextOrderService total: 0.0107 sec.
[0407 13:03.55 @doctectionpipe.py:118]  INF  Processing HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge Base Completion_1.pdf
[0407 13:03.56 @context.py:156]  INF  ImageLayoutService total: 1.7606 sec.
[0407 13:03.56 @context.py:156]  INF  AnnotationNmsService total: 0.0013 sec.
[0407 13:03.56 @context.py:156]  INF  SubImageLayoutService total: 0.0001 sec.
[0407 13:03.56 @context.py:156]  INF  PubtablesSegmentationService total: 0.0001 se

Tablas detectadas por deepdoctection: 5
  [table_1] página 6 → 4 filas × 6 cols
    Columnas: ['Dataset', '|E|', '|R|', '#Train', '#Valid', '#Test']
  [table_2] página 6 → 1 filas × 3 cols
    Columnas: ['(a)', 'is a(x,y)∧part', 'of(y,z)→part of(x,z)']
  [table_3] página 7 → 6 filas × 6 cols
    Columnas: ['Method', 'Type', 'WN18RR_MRR', 'WN18RR_H@10', 'FB15k-237_MRR', 'FB15k-237_H@10']
  [table_4] página 7 → 2 filas × 5 cols
    Columnas: ['Method', 'MRR', 'WD H@10', 'WD MRR', '++ H@10']
  [table_5] página 9 → 8 filas × 9 cols
    Columnas: ['Dataset', 'Model', '# negsE', '# negsR', 'η', 'λ', 'n', 'γ,', 'β']
  → Tablas incluidas en ground truth: 5

Procesando: Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text


[0407 13:04.17 @context.py:156]  INF  ImageLayoutService total: 1.7654 sec.
[0407 13:04.17 @context.py:156]  INF  AnnotationNmsService total: 0.0012 sec.
[0407 13:04.17 @context.py:156]  INF  SubImageLayoutService total: 0.0001 sec.
[0407 13:04.17 @context.py:156]  INF  PubtablesSegmentationService total: 0.0 sec.
[0407 13:04.17 @context.py:156]  INF  TextExtractionService total: 0.1507 sec.
[0407 13:04.17 @context.py:156]  INF  MatchingService total: 0.0067 sec.
[0407 13:04.17 @context.py:156]  INF  TextOrderService total: 0.0155 sec.
[0407 13:04.17 @doctectionpipe.py:118]  INF  Processing Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text_1.pdf
[0407 13:04.19 @context.py:156]  INF  ImageLayoutService total: 1.7611 sec.
[0407 13:04.19 @context.py:156]  INF  AnnotationNmsService total: 0.001 sec.
[0407 13:04.19 @context.py:156]  INF  SubImageLayoutService total: 0.0 sec.
[0407 13:04.19 @context.py:156]  INF  PubtablesSegmentationService total: 0.0006 sec

Tablas detectadas por deepdoctection: 2
  [table_1] página 3 → 1 filas × 2 cols
    Columnas: ['Text', 'Berlin is the capital city of Germany.']
  [table_2] página 4 → 5 filas × 4 cols
    Columnas: ['Tasks Metric', 'NYT F1-MeasureF1-Measure', 'ADE', 'Wiki-DBpedia F1-Measure']
  → Tablas incluidas en ground truth: 2

RESUMEN FINAL
  Adaptive Margin Ranking Loss for Knowledge Graph Embeddings  → 2 tablas
  Adversarial Contrastive Estimation                           → 4 tablas
  Binarized Knowledge Graph Embeddings                         → 5 tablas
  HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge → 5 tablas
  Seq2RDF- An end-to-end application for deriving Triples from → 2 tablas

Total tablas extraídas: 18


## Inspección rápida de resultados

Compara el número de tablas detectado por deepdoctection con el ground truth manual.

In [57]:
# Cargar ground truth manual para comparación
gt_manual_path = Path("pdfs_prueba/ground_truth/ground_truth_kge.json")
if gt_manual_path.exists():
    with open(gt_manual_path, 'r', encoding='utf-8') as f:
        gt_manual = json.load(f)

    manual_by_title = {d['paper_title']: d for d in gt_manual['documents']}

    print(f"{'Paper':<55} {'Manual':>8} {'DD':>8} {'Match':>8}")
    print('-' * 80)
    for doc in documents:
        title = doc['paper_title']
        dd_count = doc['num_tables']
        manual_doc = manual_by_title.get(title)
        manual_count = manual_doc['num_tables'] if manual_doc else '??'
        match = '✓' if manual_doc and dd_count == manual_doc['num_tables'] else '✗'
        print(f"{title[:55]:<55} {str(manual_count):>8} {dd_count:>8} {match:>8}")
else:
    print("Ground truth manual no encontrado, mostrando solo resultados DD:")
    for doc in documents:
        print(f"  {doc['paper_title']}: {doc['num_tables']} tablas")

Paper                                                     Manual       DD    Match
--------------------------------------------------------------------------------
Adaptive Margin Ranking Loss for Knowledge Graph Embedd        2        2        ✓
Adversarial Contrastive Estimation                             4        4        ✓
Binarized Knowledge Graph Embeddings                           5        5        ✓
HyperKG- Hyperbolic Knowledge Graph Embeddings for Know        4        5        ✗
Seq2RDF- An end-to-end application for deriving Triples        2        2        ✓


## Vista previa de una tabla extraída

Muestra la primera tabla del primer documento para verificar el formato.

In [58]:
import pandas as pd

for doc in documents:
    if doc['tables']:
        t = doc['tables'][0]
        print(f"Paper: {doc['paper_title']}")
        print(f"Tabla: {t['table_id']}  |  Página: {t['page']}")
        print(f"Columnas ({t['evaluation']['expected_cols']}): {t['evaluation']['columns']}")
        print()
        df = pd.DataFrame(t['rows'], columns=t['evaluation']['columns'])
        display(df)
        print()
        break

Paper: Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function
Tabla: table_1  |  Página: 5
Columnas (3): ['Data', 'FB15k', 'wn18']



,Data,FB15k,wn18
0,Optimizer,Adagrad,Adagrad
1,Embeddingsize,100,100
2,Epochs ξ,900000 0.1,900000 0.1
3,σ,1,1
4,λ,1,1
5,γ,30,15
6,Learningrate,0.1,0.1


## Guardar ground truth generado por deepdoctection

In [59]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(ground_truth, f, indent=2, ensure_ascii=False)

print(f"Ground truth guardado en: {OUTPUT_PATH}")
print(f"Documentos: {len(documents)}")
print(f"Total tablas: {sum(d['num_tables'] for d in documents)}")

Ground truth guardado en: pdfs_prueba/ground_truth/ground_truth_deepdoctection.json
Documentos: 5
Total tablas: 18


## Comparación detallada tabla a tabla (opcional)

Compara la estructura de cada tabla extraída con el ground truth manual.

In [60]:
if gt_manual_path.exists():
    for doc in documents:
        title = doc['paper_title']
        manual_doc = manual_by_title.get(title)
        if not manual_doc:
            print(f"\n[{title}] No encontrado en ground truth manual")
            continue

        print(f"\n{'='*70}")
        print(f"Paper: {title}")
        print(f"{'='*70}")
        print(f"  Tablas (manual): {manual_doc['num_tables']}  |  Tablas (DD): {doc['num_tables']}")

        for i, dd_table in enumerate(doc['tables']):
            print(f"\n  [{dd_table['table_id']}] página {dd_table['page']}")
            print(f"    DD     → {dd_table['evaluation']['expected_rows']} filas × {dd_table['evaluation']['expected_cols']} cols")

            if i < len(manual_doc['tables']):
                m_table = manual_doc['tables'][i]
                print(f"    Manual → {m_table['evaluation']['expected_rows']} filas × {m_table['evaluation']['expected_cols']} cols")
                row_match = dd_table['evaluation']['expected_rows'] == m_table['evaluation']['expected_rows']
                col_match = dd_table['evaluation']['expected_cols'] == m_table['evaluation']['expected_cols']
                print(f"    Filas {'✓' if row_match else '✗'}  |  Columnas {'✓' if col_match else '✗'}")
                if not col_match:
                    print(f"    Col DD:     {dd_table['evaluation']['columns']}")
                    print(f"    Col Manual: {m_table['evaluation']['columns']}")
else:
    print("Ground truth manual no disponible para comparación detallada.")


Paper: Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function
  Tablas (manual): 2  |  Tablas (DD): 2

  [table_1] página 5
    DD     → 7 filas × 3 cols
    Manual → 19 filas × 9 cols
    Filas ✗  |  Columnas ✗
    Col DD:     ['Data', 'FB15k', 'wn18']
    Col Manual: ['Dataset', 'WN18_Mean_raw', 'WN18_Mean_filter', 'WN18_Hits@10_raw', 'WN18_Hits@10_filter', 'FB15k_Mean_raw', 'FB15k_Mean_filter', 'FB15k_Hits@10_raw', 'FB15k_Hits@10_filter']

  [table_2] página 5
    DD     → 18 filas × 9 cols
    Manual → 8 filas × 3 cols
    Filas ✗  |  Columnas ✗
    Col DD:     ['Dataset', 'WN18_Mean_raw', 'WN18_Mean_filter', 'WN18_Hits@10(%)_raw', 'WN18_Hits@10(%)_filter', 'FB15k_Mean_raw', 'FB15k_Mean_filter', 'FB15k_Hits@10(%)_raw', 'FB15k_Hits@10(%)_filter']
    Col Manual: ['Data', 'FB15k', 'wn18']

Paper: Adversarial Contrastive Estimation
  Tablas (manual): 4  |  Tablas (DD): 4

  [table_1] página 8
    DD     → 9 filas × 3 cols
    Manual → 20 fila